# Bài 2 — Học máy với dữ liệu văn bản (20 Newsgroups)

**Môn học:** Học máy (Machine Learning) — Học kỳ I, năm học 2026-2027  
**Khoa Khoa học và Kỹ thuật Máy tính, Trường Đại học Bách Khoa, ĐHQG-HCM**  
**GVHD:** *[điền tên giảng viên]*  
**Nhóm:** *[tên nhóm]* — *[Họ tên – MSSV – email]* (2-3 thành viên)  
**Mã nguồn GitHub:** *[điền link repo nếu có]*

> **Hướng dẫn chạy:** chọn *Runtime → Change runtime type → T4 GPU* (khuyến nghị), sau đó *Runtime → Run all*.
> Notebook tự cài thư viện, tự tải dữ liệu công khai về máy chủ Colab, **không** mount Google Drive.

## 0. Cài đặt môi trường & nạp module

Các module Python hỗ trợ (`modules/*.py`) được ghi trực tiếp ra máy chủ Colab bằng `%%writefile` (cùng nội dung với thư mục `modules/` trong bài nộp / GitHub), nên notebook chạy độc lập mà không cần Drive.

In [ ]:
!pip -q install gensim transformers scikit-learn seaborn h5py
!mkdir -p modules reports/figures features data

In [ ]:
%%writefile modules/common.py
"""Tiện ích dùng chung cho cả 3 bài: seed, đánh giá, vẽ hình, lưu kết quả/đặc trưng."""
import json
import os
import random
import time

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from sklearn.metrics import (
    accuracy_score,
    confusion_matrix,
    precision_recall_fscore_support,
)

REPORT_DIR = "reports"
FIG_DIR = os.path.join(REPORT_DIR, "figures")
FEATURE_DIR = "features"


def ensure_dirs():
    for d in (REPORT_DIR, FIG_DIR, FEATURE_DIR, "data"):
        os.makedirs(d, exist_ok=True)


def set_seed(seed: int = 42):
    random.seed(seed)
    np.random.seed(seed)
    try:
        import torch

        torch.manual_seed(seed)
        torch.cuda.manual_seed_all(seed)
    except ImportError:
        pass


def evaluate(y_true, y_pred, average: str = "macro") -> dict:
    """Accuracy, precision, recall, F1 (average = 'macro' | 'weighted' | 'binary')."""
    p, r, f1, _ = precision_recall_fscore_support(
        y_true, y_pred, average=average, zero_division=0
    )
    return {
        "accuracy": accuracy_score(y_true, y_pred),
        "precision": p,
        "recall": r,
        "f1": f1,
    }


def save_fig(name: str, show: bool = True):
    """Lưu hình vào reports/figures/<name>.png (dùng cho báo cáo) và hiển thị."""
    ensure_dirs()
    path = os.path.join(FIG_DIR, f"{name}.png")
    plt.tight_layout()
    plt.savefig(path, dpi=140, bbox_inches="tight")
    if show:
        plt.show()
    plt.close()
    return path


def plot_confusion(y_true, y_pred, labels=None, title="Confusion matrix", name=None, normalize=True):
    cm = confusion_matrix(y_true, y_pred)
    if normalize:
        cm = cm.astype(float) / cm.sum(axis=1, keepdims=True).clip(min=1)
    n = cm.shape[0]
    size = max(5, min(14, n * 0.45))
    fig, ax = plt.subplots(figsize=(size, size * 0.85))
    im = ax.imshow(cm, cmap="Blues")
    ax.set_title(title)
    ax.set_xlabel("Dự đoán")
    ax.set_ylabel("Thực tế")
    if labels is not None and n <= 25:
        ax.set_xticks(range(n))
        ax.set_yticks(range(n))
        ax.set_xticklabels(labels, rotation=90, fontsize=7)
        ax.set_yticklabels(labels, fontsize=7)
    fig.colorbar(im, ax=ax, fraction=0.046)
    if name:
        save_fig(name)
    else:
        plt.show()


class ResultLogger:
    """Gom kết quả thí nghiệm thành bảng và lưu CSV để báo cáo dùng lại."""

    def __init__(self, name: str):
        self.name = name
        self.rows = []

    def add(self, **kwargs):
        self.rows.append(kwargs)

    def frame(self) -> pd.DataFrame:
        return pd.DataFrame(self.rows)

    def save(self):
        ensure_dirs()
        df = self.frame()
        df.to_csv(os.path.join(REPORT_DIR, f"results_{self.name}.csv"), index=False)
        return df


class Timer:
    def __enter__(self):
        self.t0 = time.time()
        return self

    def __exit__(self, *a):
        self.elapsed = time.time() - self.t0


def save_features(name: str, array: np.ndarray, fmt: str = "npy"):
    """Lưu đặc trưng vào features/<name>.npy (hoặc .h5)."""
    ensure_dirs()
    path = os.path.join(FEATURE_DIR, f"{name}.{fmt}")
    if fmt == "npy":
        np.save(path, array)
    elif fmt == "h5":
        import h5py

        with h5py.File(path, "w") as f:
            f.create_dataset("features", data=array, compression="gzip")
    else:
        raise ValueError(fmt)
    return path


def load_features(name: str, fmt: str = "npy"):
    path = os.path.join(FEATURE_DIR, f"{name}.{fmt}")
    if fmt == "npy":
        return np.load(path)
    import h5py

    with h5py.File(path, "r") as f:
        return f["features"][:]


def save_json(name: str, obj: dict):
    ensure_dirs()
    with open(os.path.join(REPORT_DIR, f"{name}.json"), "w", encoding="utf-8") as f:
        json.dump(obj, f, ensure_ascii=False, indent=2, default=float)


def download(url: str, dest: str):
    """Tải file công khai về dest (bỏ qua nếu đã có)."""
    import urllib.request

    if os.path.exists(dest):
        return dest
    os.makedirs(os.path.dirname(dest) or ".", exist_ok=True)
    print(f"Đang tải {url} ...")
    urllib.request.urlretrieve(url, dest)
    return dest

In [ ]:
%%writefile modules/text.py
"""Bài 2 - Dữ liệu văn bản: 20 Newsgroups (20 chủ đề, ~18.8k bài).

Nguồn công khai: http://qwone.com/~jason/20Newsgroups/ (tải qua sklearn.datasets.fetch_20newsgroups).
Đặc trưng truyền thống: BoW, TF-IDF (unigram / n-gram).
Đặc trưng hiện đại: GloVe (trung bình vector từ) và DistilBERT (mean pooling) -> lưu .npy.
"""
import re
from collections import Counter

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from sklearn.datasets import fetch_20newsgroups
from sklearn.feature_extraction.text import ENGLISH_STOP_WORDS, CountVectorizer, TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.naive_bayes import ComplementNB, GaussianNB, MultinomialNB
from sklearn.svm import LinearSVC

from common import ResultLogger, Timer, evaluate, save_features, save_fig

TOKEN_RE = re.compile(r"[a-z]+(?:'[a-z]+)?")


# ----------------------------------------------------------------------------
# Dữ liệu
# ----------------------------------------------------------------------------
def load_20ng(remove_meta: bool = True):
    """Trả về (train_texts, y_train, test_texts, y_test, class_names)."""
    remove = ("headers", "footers", "quotes") if remove_meta else ()
    tr = fetch_20newsgroups(subset="train", remove=remove, random_state=42)
    te = fetch_20newsgroups(subset="test", remove=remove, random_state=42)
    return tr.data, tr.target, te.data, te.target, list(tr.target_names)


def clean_text(text: str, lowercase=True, remove_stopwords=False, min_len=2) -> str:
    """Làm sạch: lowercase, bỏ URL/email/số, tokenization, (tuỳ chọn) bỏ stopwords."""
    if lowercase:
        text = text.lower()
    text = re.sub(r"https?://\S+|www\.\S+", " ", text)
    text = re.sub(r"\S+@\S+", " ", text)
    tokens = TOKEN_RE.findall(text)
    tokens = [t for t in tokens if len(t) >= min_len]
    if remove_stopwords:
        tokens = [t for t in tokens if t not in ENGLISH_STOP_WORDS]
    return " ".join(tokens)


def preprocess(texts, **kw):
    return [clean_text(t, **kw) for t in texts]


# ----------------------------------------------------------------------------
# EDA
# ----------------------------------------------------------------------------
def run_eda(train_texts, y_train, class_names, top_k: int = 20):
    n_words = np.array([len(t.split()) for t in train_texts])
    n_chars = np.array([len(t) for t in train_texts])
    empty = int((n_words == 0).sum())
    print(f"Số văn bản train: {len(train_texts)} | số lớp: {len(class_names)}")
    print(f"Văn bản rỗng sau khi bỏ header/footer/quote: {empty}")
    print(pd.Series(n_words).describe().round(1).to_string())

    counts = pd.Series(y_train).value_counts().sort_index()
    plt.figure(figsize=(10, 4))
    plt.bar(range(len(counts)), counts.values)
    plt.xticks(range(len(counts)), [class_names[i] for i in counts.index], rotation=90, fontsize=8)
    plt.title("Số văn bản theo lớp (train)")
    save_fig("text_class_dist")

    fig, axes = plt.subplots(1, 2, figsize=(12, 4))
    axes[0].hist(np.clip(n_words, 0, 1500), bins=60)
    axes[0].set_title("Phân phối số từ / văn bản (cắt ở 1500)")
    axes[0].axvline(np.median(n_words), color="r", ls="--", label=f"median={np.median(n_words):.0f}")
    axes[0].legend()
    axes[1].hist(np.log10(n_chars + 1), bins=60)
    axes[1].set_title("Phân phối log10(số ký tự)")
    save_fig("text_length_dist")

    cleaned = preprocess(train_texts, remove_stopwords=False)
    cleaned_sw = preprocess(train_texts, remove_stopwords=True)
    for name, docs in (("giữ stopwords", cleaned), ("bỏ stopwords", cleaned_sw)):
        freq = Counter(w for d in docs for w in d.split())
        print(f"\nTop {top_k} từ ({name}); kích thước từ vựng = {len(freq)}")
        top = freq.most_common(top_k)
        plt.figure(figsize=(9, 4))
        plt.bar([w for w, _ in top], [c for _, c in top])
        plt.xticks(rotation=60)
        plt.title(f"Tần suất từ phổ biến ({name})")
        save_fig("text_top_words_stop" if "giữ" in name else "text_top_words_nostop")

    # Zipf
    freq = Counter(w for d in cleaned for w in d.split())
    vals = np.array(sorted(freq.values(), reverse=True))
    plt.figure(figsize=(5, 4))
    plt.loglog(np.arange(1, len(vals) + 1), vals)
    plt.title("Luật Zipf (rank vs tần suất)")
    plt.xlabel("rank")
    plt.ylabel("tần suất")
    save_fig("text_zipf")

    # Độ dài trung bình theo lớp
    by_class = pd.DataFrame({"label": [class_names[i] for i in y_train], "n_words": n_words})
    print("\nSố từ trung bình theo lớp:")
    print(by_class.groupby("label")["n_words"].agg(["mean", "median"]).round(0).to_string())


# ----------------------------------------------------------------------------
# Đặc trưng truyền thống
# ----------------------------------------------------------------------------
def make_vectorizer(kind: str, ngram=(1, 1), max_features=50000, min_df=2, stop_words=None):
    kw = dict(ngram_range=ngram, max_features=max_features, min_df=min_df, stop_words=stop_words)
    if kind == "bow":
        return CountVectorizer(**kw)
    if kind == "tfidf":
        return TfidfVectorizer(sublinear_tf=True, **kw)
    raise ValueError(kind)


# ----------------------------------------------------------------------------
# Đặc trưng hiện đại (embedding)
# ----------------------------------------------------------------------------
_glove = None


def glove_embeddings(texts, name="glove-wiki-gigaword-100"):
    """Trung bình vector GloVe của các từ trong văn bản (OOV bị bỏ qua)."""
    global _glove
    if _glove is None:
        import gensim.downloader as api

        _glove = api.load(name)
    dim = _glove.vector_size
    out = np.zeros((len(texts), dim), dtype=np.float32)
    for i, t in enumerate(texts):
        vecs = [_glove[w] for w in TOKEN_RE.findall(t.lower()) if w in _glove.key_to_index]
        if vecs:
            out[i] = np.mean(vecs, axis=0)
    return out


def bert_embeddings(texts, model_name="distilbert-base-uncased", max_len=256, batch_size=64):
    """Embedding ngữ cảnh: mean pooling của last_hidden_state (có attention mask)."""
    import torch
    from transformers import AutoModel, AutoTokenizer

    dev = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    tok = AutoTokenizer.from_pretrained(model_name)
    model = AutoModel.from_pretrained(model_name).to(dev).eval()
    if dev.type == "cuda":
        model.half()
    # sắp theo độ dài để giảm padding
    order = np.argsort([len(t) for t in texts])
    out = np.zeros((len(texts), model.config.hidden_size), dtype=np.float32)
    with torch.no_grad():
        for s in range(0, len(texts), batch_size):
            idx = order[s:s + batch_size]
            enc = tok([texts[i] for i in idx], padding=True, truncation=True,
                      max_length=max_len, return_tensors="pt").to(dev)
            h = model(**enc).last_hidden_state
            m = enc["attention_mask"].unsqueeze(-1).to(h.dtype)
            emb = (h * m).sum(1) / m.sum(1).clamp(min=1)
            out[idx] = emb.float().cpu().numpy()
    return out


def extract_and_save(name, fn, train_texts, test_texts, **kw):
    """Chạy trích xuất embedding cho train/test và lưu .npy vào features/."""
    with Timer() as t:
        tr = fn(train_texts, **kw)
        te = fn(test_texts, **kw)
    save_features(f"text_{name}_train", tr)
    save_features(f"text_{name}_test", te)
    print(f"{name}: train {tr.shape}, test {te.shape} ({t.elapsed:.0f}s) -> features/text_{name}_*.npy")
    return tr, te


# ----------------------------------------------------------------------------
# Mô hình phân loại
# ----------------------------------------------------------------------------
def make_classifier(name: str, **kw):
    if name == "nb":
        return MultinomialNB(alpha=kw.get("alpha", 0.1))
    if name == "cnb":
        return ComplementNB(alpha=kw.get("alpha", 0.3))
    if name == "gnb":
        return GaussianNB()
    if name == "logreg":
        return LogisticRegression(max_iter=1000, C=kw.get("C", 10.0))
    if name == "linsvm":
        return LinearSVC(C=kw.get("C", 0.5), max_iter=10000)
    raise ValueError(name)


def fit_eval(clf, Xtr, ytr, Xte, yte, average="macro"):
    with Timer() as t:
        clf.fit(Xtr, ytr)
    pred = clf.predict(Xte)
    res = evaluate(yte, pred, average=average)
    res["fit_seconds"] = round(t.elapsed, 2)
    return res, pred


def finetune_bert(train_texts, y_train, test_texts, y_test, n_classes,
                  model_name="distilbert-base-uncased", max_len=192, epochs=3,
                  batch_size=32, lr=5e-5, seed=42):
    """Fine-tune đầu-cuối DistilBERT (điểm cộng pipeline học sâu)."""
    import torch
    from torch.utils.data import DataLoader, TensorDataset
    from transformers import AutoModelForSequenceClassification, AutoTokenizer, get_linear_schedule_with_warmup

    torch.manual_seed(seed)
    dev = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    tok = AutoTokenizer.from_pretrained(model_name)
    model = AutoModelForSequenceClassification.from_pretrained(model_name, num_labels=n_classes).to(dev)

    def encode(texts, labels):
        enc = tok(list(texts), padding="max_length", truncation=True, max_length=max_len, return_tensors="pt")
        return TensorDataset(enc["input_ids"], enc["attention_mask"], torch.tensor(labels))

    tr_dl = DataLoader(encode(train_texts, y_train), batch_size=batch_size, shuffle=True)
    te_dl = DataLoader(encode(test_texts, y_test), batch_size=batch_size * 2)
    opt = torch.optim.AdamW(model.parameters(), lr=lr, weight_decay=0.01)
    total = epochs * len(tr_dl)
    sched = get_linear_schedule_with_warmup(opt, int(0.1 * total), total)
    scaler = torch.amp.GradScaler("cuda", enabled=dev.type == "cuda")

    with Timer() as t:
        for ep in range(epochs):
            model.train()
            loss_sum = 0.0
            for ids, mask, lab in tr_dl:
                ids, mask, lab = ids.to(dev), mask.to(dev), lab.to(dev)
                opt.zero_grad()
                with torch.autocast(device_type=dev.type, enabled=dev.type == "cuda"):
                    loss = model(input_ids=ids, attention_mask=mask, labels=lab).loss
                scaler.scale(loss).backward()
                scaler.step(opt)
                scaler.update()
                sched.step()
                loss_sum += loss.item()
            print(f"  epoch {ep + 1}/{epochs} loss={loss_sum / len(tr_dl):.4f}")
    model.eval()
    preds = []
    with torch.no_grad():
        for ids, mask, _ in te_dl:
            logits = model(input_ids=ids.to(dev), attention_mask=mask.to(dev)).logits
            preds.append(logits.argmax(1).cpu().numpy())
    pred = np.concatenate(preds)
    res = evaluate(y_test, pred, average="macro")
    res["fit_seconds"] = round(t.elapsed, 2)
    return res, pred

In [ ]:
%%writefile modules/dl.py
"""Phần học sâu (điểm cộng): MLP PyTorch cho vector đặc trưng + vòng huấn luyện dùng chung."""
import copy

import numpy as np
import torch
import torch.nn as nn
from sklearn.model_selection import train_test_split

from common import Timer, evaluate


def get_device():
    return torch.device("cuda" if torch.cuda.is_available() else "cpu")


class MLP(nn.Module):
    def __init__(self, in_dim, n_classes, hidden=(256, 128), dropout=0.3):
        super().__init__()
        layers, d = [], in_dim
        for h in hidden:
            layers += [nn.Linear(d, h), nn.BatchNorm1d(h), nn.ReLU(), nn.Dropout(dropout)]
            d = h
        layers.append(nn.Linear(d, n_classes))
        self.net = nn.Sequential(*layers)

    def forward(self, x):
        return self.net(x)


def train_mlp(X_train, y_train, X_test, y_test, hidden=(256, 128), dropout=0.3, lr=1e-3,
              weight_decay=1e-4, epochs=60, batch_size=256, patience=8, average="macro",
              seed=42, verbose=True):
    """Huấn luyện MLP với early stopping trên 10% tập train. Trả về (metrics, model)."""
    torch.manual_seed(seed)
    dev = get_device()
    X_train = np.asarray(X_train, dtype=np.float32)
    X_test = np.asarray(X_test, dtype=np.float32)
    y_train = np.asarray(y_train)
    n_classes = int(max(y_train.max(), np.asarray(y_test).max()) + 1)

    X_tr, X_va, y_tr, y_va = train_test_split(
        X_train, y_train, test_size=0.1, stratify=y_train, random_state=seed)
    Xt, yt = torch.from_numpy(X_tr), torch.from_numpy(y_tr).long()
    Xv, yv = torch.from_numpy(X_va).to(dev), torch.from_numpy(y_va).long().to(dev)

    model = MLP(X_tr.shape[1], n_classes, hidden, dropout).to(dev)
    opt = torch.optim.AdamW(model.parameters(), lr=lr, weight_decay=weight_decay)
    loss_fn = nn.CrossEntropyLoss()
    best, best_state, bad = float("inf"), None, 0

    with Timer() as t:
        for ep in range(epochs):
            model.train()
            perm = torch.randperm(len(Xt))
            for i in range(0, len(perm), batch_size):
                idx = perm[i:i + batch_size]
                if len(idx) < 2:
                    continue
                xb, yb = Xt[idx].to(dev), yt[idx].to(dev)
                opt.zero_grad()
                loss_fn(model(xb), yb).backward()
                opt.step()
            model.eval()
            with torch.no_grad():
                vl = loss_fn(model(Xv), yv).item()
            if vl < best - 1e-4:
                best, best_state, bad = vl, copy.deepcopy(model.state_dict()), 0
            else:
                bad += 1
                if bad >= patience:
                    break
            if verbose and (ep + 1) % 10 == 0:
                print(f"  epoch {ep + 1:3d} val_loss={vl:.4f}")
    model.load_state_dict(best_state)
    model.eval()
    with torch.no_grad():
        logits = model(torch.from_numpy(X_test).to(dev))
        pred = logits.argmax(1).cpu().numpy()
    res = evaluate(y_test, pred, average=average)
    res["fit_seconds"] = round(t.elapsed, 2)
    res["epochs_run"] = ep + 1
    return res, model, pred

In [ ]:
import sys, warnings
sys.path.insert(0, 'modules')
warnings.filterwarnings('ignore')
import numpy as np, pandas as pd, matplotlib.pyplot as plt
from common import *
set_seed(42); ensure_dirs()
import text as TX
import dl

## 1. Bài toán & dữ liệu

- **Bài toán:** phân loại văn bản tin nhóm thảo luận vào **20 chủ đề** (tôn giáo, máy tính, thể thao, y tế, không gian, chính trị…).
- **Dữ liệu:** *20 Newsgroups* — http://qwone.com/~jason/20Newsgroups/ (tải qua `sklearn.datasets.fetch_20newsgroups`, ~11.3k train / ~7.5k test).
  Bỏ `headers`, `footers`, `quotes` để mô hình không "ăn gian" bằng thông tin meta.
- **Vì sao phù hợp:** nhiều nhãn (20), nội dung đa dạng, kích thước đủ lớn.

> ⚠️ Đề yêu cầu các nhóm không trùng tập dữ liệu và nên thống nhất với giảng viên. Đổi dataset: sửa `load_20ng()` trong `modules/text.py`.

In [ ]:
train_texts, y_train, test_texts, y_test, class_names = TX.load_20ng()
print(len(train_texts), len(test_texts), class_names[:5], '...')
print('--- Ví dụ ---'); print(train_texts[0][:400])

## 2. EDA

In [ ]:
TX.run_eda(train_texts, y_train, class_names)

**Nhận xét EDA** *(chỉnh theo kết quả thực tế)*:
- Số lớp tương đối cân bằng (~480–600 bài/lớp), riêng một vài lớp tôn giáo/chính trị ít hơn.
- Độ dài văn bản **lệch phải rất mạnh** (median vài trăm từ, đuôi dài hàng nghìn từ) và có một số văn bản rỗng ⇒ cần cắt (truncate) khi dùng BERT.
- Từ phổ biến nhất khi giữ stopwords là `the, to, of, and…` — không mang thông tin chủ đề ⇒ cần bỏ stopwords / dùng TF-IDF.
- Tần suất từ tuân theo **luật Zipf**: số ít từ rất phổ biến, đa số từ hiếm ⇒ cắt `min_df`, `max_features`.

## 3. Tiền xử lý văn bản (cấu hình được)

`TX.clean_text(text, lowercase, remove_stopwords, min_len)`: lowercase → bỏ URL/email → tokenization bằng regex → bỏ token ngắn → (tuỳ chọn) bỏ stopwords.
Padding chỉ áp dụng khi dùng BERT (tokenizer tự padding theo batch).

In [ ]:
print(TX.clean_text(train_texts[3])[:300]); print('...'); print(TX.clean_text(train_texts[3], remove_stopwords=True)[:300])
clean_train = TX.preprocess(train_texts, remove_stopwords=False)
clean_test = TX.preprocess(test_texts, remove_stopwords=False)
clean_train_sw = TX.preprocess(train_texts, remove_stopwords=True)
clean_test_sw = TX.preprocess(test_texts, remove_stopwords=True)

## 4. Pipeline truyền thống: BoW / TF-IDF / n-gram

Thử 3 biểu diễn × 2 chế độ stopwords × 4 bộ phân loại (Multinomial NB, Complement NB, Logistic Regression, Linear SVM).

In [ ]:
log = ResultLogger('text_traditional')
feat_cfgs = [('bow', (1, 1)), ('tfidf', (1, 1)), ('tfidf', (1, 2))]
for sw_name, (tr, te) in {'giữ stopwords': (clean_train, clean_test), 'bỏ stopwords': (clean_train_sw, clean_test_sw)}.items():
    for kind, ng in feat_cfgs:
        vec = TX.make_vectorizer(kind, ng, max_features=50000, min_df=2)
        Xtr = vec.fit_transform(tr); Xte = vec.transform(te)
        for clf_name in ['nb', 'cnb', 'logreg', 'linsvm']:
            r, _ = TX.fit_eval(TX.make_classifier(clf_name), Xtr, y_train, Xte, y_test)
            log.add(features=f'{kind}{ng}', stopwords=sw_name, classifier=clf_name, dim=Xtr.shape[1], **r)
            print(f"{kind}{ng} | {sw_name} | {clf_name:<7} acc={r['accuracy']:.4f} f1={r['f1']:.4f}")
df_trad = log.save()

In [ ]:
pv = df_trad.assign(cfg=df_trad.features + ' | ' + df_trad.stopwords).pivot(index='cfg', columns='classifier', values='f1')
import seaborn as sns
plt.figure(figsize=(7, 4)); sns.heatmap(pv, annot=True, fmt='.3f', cmap='YlGnBu'); plt.title('Macro-F1 — đặc trưng truyền thống')
save_fig('text_trad_heatmap')
df_trad.sort_values('f1', ascending=False).head(8)

## 5. Pipeline hiện đại: embedding học sâu → lưu `.npy`

Hai kiểu embedding:
1. **GloVe** (`glove-wiki-gigaword-100`, huấn luyện sẵn): trung bình vector các từ → 100 chiều.
2. **DistilBERT** (`distilbert-base-uncased`): embedding ngữ cảnh, mean pooling theo attention mask → 768 chiều.

Embedding được lưu vào `features/text_*_{train,test}.npy` và dùng làm đầu vào bộ phân loại.

In [ ]:
glove_tr, glove_te = TX.extract_and_save('glove100', TX.glove_embeddings, train_texts, test_texts)

In [ ]:
bert_tr, bert_te = TX.extract_and_save('distilbert', TX.bert_embeddings, train_texts, test_texts, max_len=256, batch_size=64)
save_features('text_y_train', np.asarray(y_train)); save_features('text_y_test', np.asarray(y_test))

In [ ]:
# Nạp lại từ file .npy để chứng minh quy trình tái sử dụng đặc trưng
emb = {
    'glove100': (load_features('text_glove100_train'), load_features('text_glove100_test')),
    'distilbert': (load_features('text_distilbert_train'), load_features('text_distilbert_test')),
}
log2 = ResultLogger('text_embeddings')
for name, (Xtr, Xte) in emb.items():
    for clf_name in ['gnb', 'logreg', 'linsvm']:
        clf = TX.make_classifier(clf_name, C=1.0) if clf_name != 'linsvm' else TX.make_classifier('linsvm', C=0.1)
        if clf_name != 'gnb':
            from sklearn.pipeline import make_pipeline
            from sklearn.preprocessing import StandardScaler
            clf = make_pipeline(StandardScaler(), clf)
        r, _ = TX.fit_eval(clf, Xtr, y_train, Xte, y_test)
        log2.add(features=name, classifier=clf_name, dim=Xtr.shape[1], **r)
        print(f"{name} | {clf_name:<7} acc={r['accuracy']:.4f} f1={r['f1']:.4f}")
df_emb = log2.save(); df_emb

## 6. (Điểm cộng) Pipeline học sâu

**6a.** MLP (PyTorch) trên embedding DistilBERT đã lưu.
**6b.** **Fine-tune đầu-cuối DistilBERT** (3 epoch, `max_len=192`) — toàn bộ mạng được tối ưu cùng lúc bằng gradient descent.

In [ ]:
log3 = ResultLogger('text_deep')
for name in ['glove100', 'distilbert']:
    Xtr, Xte = emb[name]
    mu, sd = Xtr.mean(0), Xtr.std(0) + 1e-6
    r, _, _ = dl.train_mlp((Xtr - mu) / sd, y_train, (Xte - mu) / sd, y_test, hidden=(512, 256), dropout=0.4, verbose=False)
    log3.add(method=f'MLP trên {name}', **r); print(f'MLP trên {name}:', {k: round(v, 4) for k, v in r.items()})

In [ ]:
import torch
if torch.cuda.is_available():
    r_ft, pred_ft = TX.finetune_bert(train_texts, y_train, test_texts, y_test, n_classes=len(class_names), epochs=3, max_len=192)
else:
    # Không có GPU: fine-tune trên tập con nhỏ để notebook vẫn chạy (kết quả sẽ thấp hơn)
    print('Không có GPU -> fine-tune trên 2000 mẫu train, 1 epoch (chỉ để minh hoạ). Hãy bật GPU để có kết quả đầy đủ.')
    rng = np.random.default_rng(0); sub = rng.choice(len(train_texts), min(2000, len(train_texts)), replace=False)
    r_ft, pred_ft = TX.finetune_bert([train_texts[i] for i in sub], np.asarray(y_train)[sub], test_texts[:1000], np.asarray(y_test)[:1000],
                                     n_classes=len(class_names), epochs=1, max_len=128)
log3.add(method='Fine-tune DistilBERT', **r_ft)
print({k: round(v, 4) for k, v in r_ft.items()})
df_deep = log3.save(); df_deep

## 7. Tổng hợp & phân tích

In [ ]:
best_trad = df_trad.sort_values('f1', ascending=False).iloc[0]
best_emb = df_emb.sort_values('f1', ascending=False).iloc[0]
summary = pd.DataFrame([
    {'approach': 'Truyền thống', 'method': f"{best_trad['features']} + {best_trad['classifier']} ({best_trad['stopwords']})", 'accuracy': best_trad['accuracy'], 'f1': best_trad['f1']},
    {'approach': 'Embedding + ML cổ điển', 'method': f"{best_emb['features']} + {best_emb['classifier']}", 'accuracy': best_emb['accuracy'], 'f1': best_emb['f1']},
] + [{'approach': 'Học sâu', 'method': r['method'], 'accuracy': r['accuracy'], 'f1': r['f1']} for _, r in df_deep.iterrows()])
summary.to_csv('reports/results_text_summary.csv', index=False)
plt.figure(figsize=(9, 4)); plt.barh(summary['method'], summary['f1'], color=plt.cm.viridis(np.linspace(0.1, 0.9, len(summary))))
plt.xlabel('Macro-F1'); plt.title('So sánh các cách tiếp cận (20 Newsgroups)'); plt.xlim(0, 1)
for i, v in enumerate(summary['f1']): plt.text(v + 0.005, i, f'{v:.3f}', va='center')
save_fig('text_summary_bar'); summary

In [ ]:
# Phân tích lỗi của mô hình truyền thống tốt nhất
sw = best_trad['stopwords'] == 'bỏ stopwords'
tr, te = (clean_train_sw, clean_test_sw) if sw else (clean_train, clean_test)
kind, ng = ('bow', (1, 1)) if best_trad['features'].startswith('bow') else ('tfidf', (1, 2) if '(1, 2)' in best_trad['features'] else (1, 1))
vec = TX.make_vectorizer(kind, ng, max_features=50000, min_df=2)
Xtr = vec.fit_transform(tr); Xte = vec.transform(te)
r, pred = TX.fit_eval(TX.make_classifier(best_trad['classifier']), Xtr, y_train, Xte, y_test)
plot_confusion(y_test, pred, labels=class_names, title='Confusion matrix — mô hình truyền thống tốt nhất', name='text_confusion_best')
from sklearn.metrics import classification_report
rep = pd.DataFrame(classification_report(y_test, pred, target_names=class_names, output_dict=True)).T.iloc[:-3]
print(rep.sort_values('f1-score').head(6).round(3))
cm = pd.crosstab(pd.Series(np.array(class_names)[y_test], name='thực tế'), pd.Series(np.array(class_names)[pred], name='dự đoán'))
cm = cm.reindex(index=class_names, columns=class_names, fill_value=0)
pairs = [(a, b, cm.loc[a, b]) for a in cm.index for b in cm.columns if a != b]
print('\nCặp nhầm lẫn nhiều nhất:'); [print(f'  {a} -> {b}: {n}') for a, b, n in sorted(pairs, key=lambda x: -x[2])[:6]]

## 8. Kết luận *(chỉnh theo kết quả thực tế)*

- **Truyền thống:** TF-IDF (unigram/bigram) + Linear SVM/Logistic Regression thường rất mạnh trên 20 Newsgroups; Naive Bayes nhanh nhưng thấp hơn.
- **Bỏ stopwords / n-gram:** *…*
- **Embedding GloVe (trung bình):** mất thông tin thứ tự và từ hiếm ⇒ thường *thấp hơn* TF-IDF trên bài toán phân loại chủ đề.
- **DistilBERT (đóng băng):** *…*; **fine-tune** thường cho kết quả cao nhất nhưng tốn tài nguyên GPU.
- **Nhầm lẫn:** các nhóm gần nghĩa (`comp.*`, `talk.politics.*`, `religion`/`atheism`) bị lẫn với nhau.
- **Hạn chế:** cắt văn bản ở 192–256 token, chưa tinh chỉnh siêu tham số quy mô lớn, một lần chia train/test cố định.

Đặc trưng embedding đã lưu ở `features/text_*.npy`; bảng kết quả ở `reports/results_text_*.csv`.